# Lesson 24: Pulse Voltammetry II: Differential Pulse and Square Wave
Every technique so far has produced a **wave**: the Nernst equation sets limiting behaviour at both extremes. The two techniques in this lesson return the same information as a **peak** instead, and they do it by counting the same electrons twice.

Today's topics:
- Differential pulse voltammetry and the differential measurement
- Square wave voltammetry, and cancelling the charging current
- Reading information straight off a peak

**Reference:**
- Sections 7.3 and 7.4 in *Electrochemical Methods: Fundamentals and Applications*, Allen J. Bard & Larry R. Faulkner.
---

## Configure Environment

In [ ]:
import sys
if '../Resources' not in sys.path:
    sys.path.append('../Resources')     # so that echem.py can be found

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

import echem                            # the shared toolkit for this course
from echem import R, T, F, f, WIDE      # constants and the two-panel figure size
echem.use_style()                       # course-wide graph formatting

# Experimental settings shared by the figures in this lesson
POTENTIAL = np.linspace(-0.3, 0.3, 151)     # V vs E0'
TIME = np.linspace(0.005, 2, 500)           # s

# 4. Differential Pulse Voltammetry

Differential pulse voltammetry (DPV) is a modern technique that is highly sensitive and capable of providing excellent data.

#### Differential Pulse Waveform

The waveform consists of a single pulse waveform superimposed on a staircase waveform.

- $n$ : number of steps
- $E_i$ : initial potential; typically chosen to be $E_{OC}$ 
- $t_s$ : pre-electrolysis time that establishes **apparent concentrations** at electrode surface
- $\Delta E_s$ : height of staircase steps
- $\tau_s$ : current sampling time, relative to start of $t_s$ 
- $t_p$ : duration of voltage pulse
- $\Delta E_p$ : height of the voltage pulse
- $\tau_p$ : current sampling time, relative to start of $t_p$ 
- $\nu$ : the effective scan rate, defined as:

 $
 \large \nu = \frac{\Delta E_s}{t_s + t_p}
 $

The experiment relies on **differential currents**, which means that the measured signal becomes:

$
\large \Delta i = i(\tau_p)-i(\tau_s)
$

In [ ]:
@interact(e_i=widgets.FloatText(value=0.0, step=0.01, description=r'$E_i$'),
          e_step=widgets.FloatText(value=0.01, step=0.01, description=r'$\Delta E_s$'),
          t_step=widgets.FloatText(value=3., step=0.01, description=r'$t_s$'),
          e_pulse=widgets.FloatText(value=0.01, step=0.01, description=r'$\Delta E_p$'),
          t_pulse=widgets.FloatText(value=1., step=0.01, description=r'$t_p$'),
          tau_s=widgets.FloatText(value=2.5, step=0.1, description=r'$\tau_s$'),
          tau_p=widgets.FloatText(value=0.5, step=0.1, description=r'$\tau_p$'),
          n=widgets.BoundedIntText(value=4, min=2, max=12, description=r'$n$'))
def plot_dpv_waveform(e_i, e_step, t_step, tau_s, e_pulse, t_pulse, tau_p, n):
    '''Differential pulse: a pulse train riding on a staircase.

    The red markers are sampled at the end of each pulse, the green markers just
    before the next one.  The DIFFERENCE between them is the analytical signal,
    which is why the output is a peak rather than a wave.
    '''
    if min(t_step, t_pulse) <= 0:
        print('Both times must be greater than zero.')
        return
    cycle = t_step + t_pulse
    t, e, base = echem.dpv_waveform(e_initial=e_i, e_step=e_step, e_pulse=e_pulse,
                                    t_step=t_step, t_pulse=t_pulse, steps=n)

    fig = plt.figure()
    ax = plt.subplot()
    ax.set_xlabel('t /s')
    ax.set_ylabel(r'E / V vs. $E_{OC}$')

    ax.annotate('', fontsize=6, xy=(cycle + t_pulse, e_i + e_pulse*0.5), xycoords='data',
                xytext=(2*cycle, e_i + e_pulse*0.5), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$t_s$', (cycle*1.5, e_i + e_pulse*0.6), fontsize=6)

    ax.annotate('', fontsize=6, xy=(t_step*0.5, e_i), xycoords='data',
                xytext=(t_step*0.5, e_i + e_pulse), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$\Delta E_s$', (t_step*0.6, e_i + e_pulse*0.5), fontsize=6)

    ax.annotate('', fontsize=6, xy=(cycle + t_pulse, e_i + (e_pulse + e_step)*1.2),
                xycoords='data', xytext=(cycle, e_i + (e_pulse + e_step)*1.2),
                textcoords='data', arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$t_p$', (cycle + t_pulse*0.5, e_i + (e_pulse + e_step)*1.3), fontsize=6)

    ax.annotate('', fontsize=6, xy=(cycle + t_pulse*1.3, e_i + e_step), xycoords='data',
                xytext=(cycle + t_pulse*1.3, e_i + e_step + e_pulse), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$\Delta E_p$', (cycle + t_pulse*1.4, e_i + e_step + e_pulse*0.5), fontsize=6)

    ax.plot(t, e, 'blue')
    ax.plot(t, base, linestyle=':', color='gray')
    for step in range(1, n):
        ax.scatter([tau_p + step*cycle], [e_i + step*e_step + e_pulse], color='red', s=10)
        ax.scatter([t_pulse + tau_s + step*cycle], [e_i + step*e_step], color='green', s=10)

As can be inferred from the waveform, the output of DPV should be related both NPV and staircase voltammetry. The most important benefit of DPV arises due to the use of a differential measurement: electrons (our analytical signal) are effectively counted twice. The species is first reduced, then immediately oxidized. 

<div class="alert alert-block alert-info">
<b>Activity:</b> Assume that you have solution containing only the oxidized form of a redox couple. Sketch the diffusion layer that would exist at the pulse and step sampling times as the voltage is increased to, and beyond, the formal potential.
</div>

If one were to record the current continually through the experiment, the results would look quite terrible:

<img src='../Resources/images/dpv-measured.png' width=25%>

Once processed, plots of $\Delta i$ vs $E$ provide clean access to the desired information:

<img src='../Resources/images/dpv-processed.png' width=25%>

One key feature to notice is that the all previous results yielded a graph that is conventionally described as a **wave**: the Nernst equation yields limiting behavior at high and low potentials. This new form of information instead yields the information in the format of a **peak**. For many people, this is vastly preferable - peaks are easier to analyze!

The location of the peak is related to the $E_{1/2}$ values that we have seen throughout the voltammetry discussions to this point. An extra term is added in for the resolution of the experiment.

#### Reading the peak

That extra term is worth writing down, because both numbers you take off a DPV peak carry one.

The peak sits half a pulse short of the half-wave potential, measured against the **staircase** potential rather than the pulse:

$
\large E_p = E_{1/2} - \frac{\Delta E_p}{2}
$

which makes sense from the waveform: $\Delta i$ is largest when the base and pulse potentials straddle $E_{1/2}$ , and they straddle it evenly when the base sits $\Delta E_p/2$ short.

The **width at half height** carries the electron count:

$
\large W_{1/2} = \frac{3.52RT}{nF} \xrightarrow{\ \ 298\ \mathrm{K}\ \ } \frac{90.4}{n}\ \mathrm{mV}
$

Where:
- $W_{1/2}$ : full width of the peak at half its maximum height, V
- $n$ : number of electrons transferred

Take care: this holds only in the limit of a *small* pulse, which the next section is about. Notice also what it is made of. This is the peak-shaped cousin of the 59.2/ $n$ mV per decade you have checked against since [Lesson 9](L09-Thermodynamics-I.ipynb): the same $RT/nF$ , read off a different feature.

<div class="alert alert-block alert-success">
<b>Half-height width</b>: $W_{1/2} = 3.52RT/nF$ , or 90.4/ $n$ mV at 298 K, for a reversible couple and a small pulse. A one-electron peak that comes back much wider than this is telling you about kinetics, uncompensated resistance or two couples you have not resolved. It is not telling you about $n$ .
</div>

<div class="alert alert-block alert-danger">
<b>Check your units!</b> Satisfy yourself that $3.52RT/nF$ really does come out in volts, and then evaluate it for $n$ = 1 at 298 K. If you do not land within a millivolt of 90.4 mV, find the slip now rather than in the exam: this is a number worth knowing by heart.
</div>

#### Sensitivity against resolution

The two equations are in tension. $W_{1/2}$ is $90.4/n$ mV only as $\Delta E_p \rightarrow 0$ , and a vanishing pulse gives a vanishing $\Delta i$ . Growing the pulse buys signal and spends resolution.

The widget puts two one-electron couples in one solution, separated by a gap you control. Every current in it is the superposition sum of [Lesson 23](L23-PulseVoltammetry-I.ipynb): a DPV cycle is a base step and a pulse step sampled once each, so `echem.dpv_response` is a call to `echem.step_train_current` and a subtraction.

There is a familiar face in the left panel. The green curve, sampled just before each pulse, is the **staircase voltammogram** of Lesson 23, give or take the disturbance left by the intervening pulses. The red curve is that experiment interrupted by a pulse. Neither is pleasant to read alone; their difference is the right-hand panel.

<div class="alert alert-block alert-info">
<b>Activity:</b> Before you touch the sliders, predict what happens to the pair of peaks as $\Delta E_p$ grows from 5 mV to 100 mV. Write down both halves of the answer: what you expect to gain, and what you expect to lose.
</div>

- As $\Delta E_p$ grows, I expect to gain: ANSWER
- As $\Delta E_p$ grows, I expect to lose: ANSWER
- The setting I would choose for a trace analyte with no interferences: ANSWER

In [ ]:
@interact(e_pulse=widgets.FloatSlider(value=0.05, min=0.005, max=0.15, step=0.005,
                                      readout_format='.3f', description=r'$\Delta E_p$ /V'),
          separation=widgets.FloatSlider(value=0.15, min=0.02, max=0.40, step=0.01,
                                         readout_format='.2f', description=r'gap /V'),
          t_pulse=widgets.FloatText(value=0.05, step=0.01, description=r'$t_p$ /s'))
def plot_dpv_output(e_pulse, separation, t_pulse):
    """Two couples, one scan: what a bigger pulse buys, and what it costs."""
    if t_pulse <= 0:
        print('The pulse time must be greater than zero.')
        return

    scan = np.arange(-0.45, 0.45, 0.002)
    formals = (-separation/2, separation/2)

    # Two independent couples in the same solution: run the calculation for
    # each and add the currents, because diffusion is linear.
    delta = np.zeros_like(scan)
    at_step = np.zeros_like(scan)
    at_pulse = np.zeros_like(scan)
    for e_formal in formals:
        d, i_step, i_pulse = echem.dpv_response(scan, e_pulse=e_pulse, t_step=0.5,
                                                t_pulse=t_pulse, conc_total=1e-6,
                                                e_formal=e_formal)
        delta = delta + d
        at_step = at_step + i_step
        at_pulse = at_pulse + i_pulse

    fig = plt.figure(figsize=WIDE)
    ax1 = plt.subplot(121)
    ax1.set_title('The two sampled currents')
    ax2 = plt.subplot(122)
    ax2.set_title(r'$\Delta i$ vs $E$')

    # The measurement itself.  The green curve is essentially the staircase
    # voltammogram of Lesson 23; the red one is the same thing interrupted by a
    # pulse.  Everything the technique reports is the gap between them.
    ax1.plot(scan, at_pulse*1e6, color='red', label=r'$i(\tau_p)$, end of pulse')
    ax1.plot(scan, at_step*1e6, color='green',
             label=r'$i(\tau_s)$, before pulse (the staircase)')
    ax1.set_xlabel(r"$E$ /V vs $E^{0'}$")
    ax1.set_ylabel(r'$i$ /$\mu$A')
    ax1.legend(fontsize=6, loc=2)

    ax2.plot(scan, delta*1e6, color=echem.color(0))
    for e_formal in formals:
        ax2.axvline(e_formal - e_pulse/2, linestyle=':', color='black')
    ax2.set_ylim(top=delta.max()*1e6*1.18)
    ax2.annotate(r'$E_{1/2} - \Delta E_p/2$',
                 (formals[0] - e_pulse/2, delta.max()*1e6*1.07), fontsize=6,
                 ha='center')
    ax2.set_xlabel(r"$E$ /V vs $E^{0'}$")
    ax2.set_ylabel(r'$\Delta i$ /$\mu$A')

    # One couple on its own, so the width readout is not confused by the pair.
    single, _, _ = echem.dpv_response(scan, e_pulse=e_pulse, t_step=0.5,
                                      t_pulse=t_pulse, conc_total=1e-6)
    half = np.where(single >= single.max()/2)[0]
    width = 1000*(scan[half[-1]] - scan[half[0]])
    maxima = np.sum((delta[1:-1] > delta[:-2]) & (delta[1:-1] >= delta[2:]))

    print(f'one couple alone: peak {single.max()*1e6:6.3f} uA, '
          f'W(1/2) = {width:5.1f} mV   (3.52RT/F = {1000*3.52*R*T/F:.1f} mV)')
    print(f'the pair, {1000*separation:.0f} mV apart: '
          f'{"RESOLVED, 2 peaks" if maxima == 2 else f"NOT resolved, {maxima} peak"}')

Both effects are real, and both are large.

Growing $\Delta E_p$ from 5 mV to 100 mV multiplies the peak height by more than fifteen, very nearly linearly at the small-pulse end. This is why nobody runs DPV with a tiny pulse: you would be throwing away the sensitivity the technique exists to deliver.

The cost is in the width readout. At 5 mV the peak comes back within a millivolt or two of 90 mV, so $3.52RT/nF$ is doing its job. By 100 mV it is a third wider, and pushing the pulse to 150 mV, comparable with the separation itself, merges the pair into one broad feature that anyone would report as a single species.

There is no correct setting, only a choice that depends on the sample:

- one analyte, nothing nearby: a large pulse, 50 to 100 mV, and take the sensitivity
- couples within roughly 100 mV of each other: 25 mV or less, and accept the smaller signal
- either way, **report $\Delta E_p$ with the data**. $E_p$ is offset from $E_{1/2}$ by $\Delta E_p/2$ , so a peak potential quoted without its pulse height is a number nobody else can use.

# 5. Square Wave Voltammetry

Square wave voltammetry (SWV) takes the concept behind DPV and brings it into the modern digital era. The waveform consists of a squared-off sine-wave function that is superimposed on a staircase. The analytical signal is again a difference in current, this time between the forwards and backwards voltage waves, but the systematic approach yields improved results.

#### Square Wave Waveform

The waveform consists of a square wave superimposed on a staircase waveform.

- $n$ : number of cycles
- $E_i$ : initial potential; typically chosen to be $E_{OC}$ 
- $t$ : cycle time 
- $\Delta E_s$ : height of staircase steps
- $\Delta E_p$ : height of the voltage pulse
- $\tau$ : current sampling time
- $\nu$ : the effective scan rate, defined as:

 $
 \large \nu = \frac{\Delta E_s}{t}
 $

The experiment again relies on **differential currents**:

$
\large \Delta i = i(\tau_2)-i(\tau_1)
$

In [ ]:
@interact(e_i=widgets.FloatText(value=0.0, step=0.01, description=r'$E_i$'),
          e_step=widgets.FloatText(value=0.01, step=0.01, description=r'$\Delta E_s$'),
          e_pulse=widgets.FloatText(value=0.01, step=0.01, description=r'$\Delta E_p$'),
          t_step=widgets.FloatText(value=2., step=0.01, description=r'$t$'),
          tau=widgets.FloatText(value=0.5, step=0.1, description=r'$\tau$'),
          n=widgets.BoundedIntText(value=4, min=2, max=12, description=r'$n$'))
def plot_swv_waveform(e_i, e_step, e_pulse, t_step, tau, n):
    '''Square wave: a square wave summed onto a staircase.

    Forward (red) and reverse (green) halves are sampled and subtracted.  Being
    equal and opposite, the double layer charging current cancels between them,
    which is what allows the very high effective scan rates.
    '''
    if t_step <= 0 or not 0 < tau < t_step:
        print('The step time must be positive and tau must fall inside it.')
        return
    t, e, base = echem.swv_waveform(e_initial=e_i, e_step=e_step,
                                    e_pulse=e_pulse, t_step=t_step, steps=n)

    fig = plt.figure()
    ax = plt.subplot()
    ax.set_xlabel('t /s')
    ax.set_ylabel(r'E / V vs. $E_{OC}$')

    ax.annotate('', fontsize=6, xy=(0, e_i + e_pulse*1.1), xycoords='data',
                xytext=(t_step, e_i + e_pulse*1.1), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$t$', (t_step*0.5, (e_i + e_pulse)*1.2), fontsize=6)

    ax.annotate('', fontsize=6, xy=(t_step*0.15, e_i), xycoords='data',
                xytext=(t_step*0.15, e_i + e_pulse), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$\Delta E_p$', (t_step*0.2, e_i + e_pulse*0.5), fontsize=6)

    ax.annotate('', fontsize=6, xy=(t_step*0.95, e_i), xycoords='data',
                xytext=(t_step*0.95, e_i + e_step), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$\Delta E_s$', (t_step*0.7, e_i + e_step*0.5), fontsize=6)

    ax.plot(t, e, 'blue')
    ax.plot(t, base, linestyle=':', color='gray')
    for step in range(n):
        ax.scatter([tau + step*t_step], [e_i + e_pulse + step*e_step], color='red', s=10)
        ax.scatter([tau + t_step/2 + step*t_step], [e_i - e_pulse + step*e_step],
                   color='green', s=10)

<div class="alert alert-block alert-info">
<b>Activity:</b> The forward and reverse pulses of a square wave are equal in size and opposite in direction. On one set of axes, for a single cycle, sketch:
<br><br>
1. the double layer charging current, and
<br>
2. the Faradaic current
<br><br>
Mark where you would sample in each half cycle. Then answer: the subtraction removes one of these two and keeps the other. Which is which, and what is it about their <i>shapes</i> that lets the trick work?
</div>

- Removed by the subtraction: ANSWER
- Kept by the subtraction: ANSWER
- Their shapes differ in that: ANSWER

The systematic approach to perturbing the system and measuring the current response yields much cleaner results:

<img src='../Resources/images/swv-measured.png' width=30%>

Processing of the data involves converting the y-axis from measured current into differential current between adjacent positive/negative portions of the square wave. The expected behavior for a reversible system can be provided in dimensionless current units. In the diagram below:

- $\psi_f$ : the current measured on the forward sweep/step
- $\psi_r$ : the current measured on the reverse sweep/step
- $\Delta \Psi$ : the differential current that we seek

<img src='../Resources/images/swv1.png' width=40%>

#### Where the peak comes from

That figure is worth generating rather than reading, because the three curves in it do not behave alike.

`echem.swv_response` builds the square wave as a step programme, hands it to the same superposition sum that produced NPV, staircase and DPV, and reports the result in the dimensionless $\psi$ defined above.

In [ ]:
@interact(e_pulse=widgets.FloatSlider(value=0.025, min=0.005, max=0.100, step=0.005,
                                      readout_format='.3f', description=r'$\Delta E_p$ /V'),
          e_step=widgets.FloatSlider(value=0.005, min=0.001, max=0.020, step=0.001,
                                     readout_format='.3f', description=r'$\Delta E_s$ /V'),
          frequency=widgets.FloatText(value=50.0, step=10.0, description=r'$f$ /Hz'))
def plot_swv_output(e_pulse, e_step, frequency):
    """The forward and reverse components, and the difference that is the signal."""
    if min(e_step, frequency) <= 0:
        print('The step height and the frequency must both be greater than zero.')
        return

    scan = np.arange(-0.3, 0.3, e_step)
    psi_f, psi_r, delta_psi = echem.swv_response(scan, e_pulse=e_pulse,
                                                 t_step=1.0/frequency,
                                                 conc_total=1e-6)

    fig = plt.figure()
    ax = plt.subplot()
    ax.axhline(0, linestyle=':', color='gray', linewidth=0.5)
    ax.plot(scan, psi_f, color=echem.color(0), label=r'$\psi_f$  forward')
    ax.plot(scan, psi_r, color=echem.color(1), label=r'$\psi_r$  reverse')
    ax.plot(scan, delta_psi, color='black', label=r'$\Delta \Psi$  signal')
    ax.set_xlabel(r"$E$ /V vs $E^{0'}$")
    ax.set_ylabel(r'$\psi$')
    ax.legend(fontsize=6, loc=2)

    k = np.argmax(delta_psi)
    half = np.where(delta_psi >= delta_psi[k]/2)[0]
    print(f'peak at E = {1000*scan[k]:+5.1f} mV vs E0\'   |   '
          f'W(1/2) = {1000*(scan[half[-1]] - scan[half[0]]):5.1f} mV'
          f'   (3.52RT/F = {1000*3.52*R*T/F:.1f} mV)')
    print(f'psi_f = {psi_f[k]:+.4f}, psi_r = {psi_r[k]:+.4f}, '
          f'delta_psi_p = {delta_psi[k]:.4f}'
          f'   |   the difference is {delta_psi[k]/psi_f[k]:.2f} times psi_f alone')

Three things to have noticed.

 $\psi_f$ and $\psi_r$ are not two versions of the same curve. On the forward half cycle the electrode drives the reaction; on the reverse half it drives it back, on material the forward half has just made and that is still sitting at the surface. **How much** it drives back depends on the amplitude, and there is a threshold. Below roughly 15 mV the reverse pulse is too shallow to reverse anything: $\psi_r$ stays positive, merely smaller, and the subtraction takes a difference. Above it $\psi_r$ changes sign, and subtracting a negative number *adds* magnitudes. Watch the printed ratio $\Delta \Psi_p / \psi_f$ cross 1.0 as you sweep through that point: below the crossing you are measuring a difference, above it you are genuinely counting the same electrons twice.

The peak sits at $E^{0'}$ , not half a pulse short of it. The square wave straddles the staircase potential evenly, so the DPV offset cancels and there is nothing to correct for. This is the $E_{max}$ expression below, with $D_O = D_R$ .

And at small amplitude the peak is again $90.4/n$ mV wide. Two different waveforms, one shared limit, and it should not surprise you: at a small enough perturbation both experiments do nothing but report the slope of the same Nernstian wave.

One caution. All of this assumes a **reversible** couple. Once electron transfer is slow the reverse component collapses first, so $\psi_r/\psi_f$ falling well below its reversible value is a useful warning. Calculating that case needs a numerical solution of an integral equation and is beyond the scope of this course; treat these curves as the reversible reference that data is judged against.

SWV comes with many substantial benefits:

#### Facilitated Analysis Using Peaks

The peak location will correspond to the $E_{1/2}$ measured in sampled current voltammetry. Therefore, information can be readily obtained by simply picking the maximum in the signal:

$
\large E_{max} = E^{0'} + \frac{RT}{nF}ln\left(\sqrt{\frac{D_R}{D_O}}\right)
$

This makes it very fast and easy to analyze mass transfer effects, to determine the influence of chemical reactions, and to judge the electrochemical reversibility of the system.

The maximum peak current can be described as:

$
\large \Delta i_{max} = \frac{nFA\sqrt{D_O}C_O^*}{\sqrt{\pi t_p}}\Delta \psi_p
$

The $\psi_p$ parameter is a function of the number of electrons associated with the electrochemical reaction and the experimental conditions. Tabulated values are available:

<img src='../Resources/images/swv-table.png' width=20%>

#### Extremely High Scan Rates

The pairing of equivalent forwards and reverse voltage steps effectively eliminates the effect of double layer capacitance. When paired with a system that provides high rates of mass transport, such as ultramicroelectrodes, one can obtain excellent data over a surprising range of scan rates. For example:


<img src='../Resources/images/swv-ume.png' width=50%>

#### Why the charging current cancels

That claim has now been made twice without support: pairing equal and opposite pulses "effectively eliminates the effect of double layer capacitance". It is worth two minutes, because the answer is *yes, but not for the reason the sentence suggests*.

Recall from [Lesson 7](L07-Resistance-DoubleLayer.ipynb) that a step of size $\Delta E$ across a cell with uncompensated resistance $R_u$ and double layer capacitance $C_d$ drives a non-Faradaic charging current:

$
\large i_c(t) = \frac{\Delta E}{R_u}e^{-t/R_uC_d}
$

Where:
- $R_uC_d$ : the **cell time constant**, s
- $\Delta E$ : the size of the potential step, V

Set that beside the Faradaic current, which decays as $t^{-1/2}$ . The two die at wildly different rates, and it is that difference, rather than the subtraction, that does most of the work.

In [ ]:
@interact(r_u=widgets.FloatText(value=100.0, step=10.0, description=r'$R_u$ /$\Omega$'),
          c_d=widgets.FloatText(value=1.4e-6, step=1e-7, description=r'$C_d$ /F'),
          frequency=widgets.FloatLogSlider(value=500.0, base=10, min=1, max=4, step=0.1,
                                           readout_format='.0f', description=r'$f$ /Hz'))
def plot_charging_vs_faradaic(r_u, c_d, frequency):
    """The charging current dies exponentially. The Faradaic current does not."""
    if min(r_u, c_d, frequency) <= 0:
        print('All three values must be greater than zero.')
        return

    area = 0.07                              # a 3 mm disk, cm^2
    conc = 1e-8                              # 10 micromolar: the trace regime SWV is for
    half = 0.5/frequency                     # one half cycle, s
    t = np.linspace(half/1000, half, 1000)

    tau_cell = echem.cell_time_constant(r_u, c_d)
    i_c = echem.charging_current(t, 0.025, r_u, c_d)
    i_f = echem.cottrell(t, conc, area=area)

    fig = plt.figure(figsize=WIDE)
    ax1 = plt.subplot(121)
    ax1.set_title('One half cycle')
    ax2 = plt.subplot(122)
    ax2.set_title('Same currents, log scale')
    for ax in (ax1, ax2):
        ax.plot(t*1e3, i_c*1e6, color=echem.color(1), label=r'$i_c$  charging')
        ax.plot(t*1e3, i_f*1e6, color=echem.color(0), label=r'$i_f$  Faradaic')
        ax.axvline(half*1e3, linestyle=':', color='red')
        ax.set_xlabel(r'$t$ /ms')
        ax.set_ylabel(r'$i$ /$\mu$A')
    ax1.set_ylim(0, 5*i_f[-1]*1e6)
    ax1.annotate('sample here ', (half*1e3, 4.4*i_f[-1]*1e6), fontsize=6, color='red',
                 ha='right')
    ax2.set_yscale('log')
    ax2.set_ylim(bottom=i_f[-1]*1e-3*1e6)      # 3 decades below the sampled Faradaic current
    ax2.legend(fontsize=6, loc=1)

    print(f'R_u C_d = {tau_cell*1e3:.3f} ms   |   half cycle = {half*1e3:.3f} ms '
          f'= {half/tau_cell:.1f} time constants')
    print(f'at the sampling instant:  i_c = {i_c[-1]*1e6:9.3e} uA, '
          f'i_f = {i_f[-1]*1e6:.3f} uA,  i_c/i_f = {i_c[-1]/i_f[-1]:.2e}'
          f'   ->  {"usable" if half > 5*tau_cell else "TOO FAST for this cell"}')

The log panel is the one to look at. The charging current falls off a cliff, because that is what an exponential does; the Faradaic current merely sags, because $t^{-1/2}$ is a lazy function. They cross, and everything to the right of the crossing is a usable sampling window.

So the honest version of the claim has two halves. The subtraction *does* cancel whatever charging current remains, since the forward and reverse contributions are equal and opposite. But the reason square wave voltammetry works at all is the **waiting**: sampling at the end of each half cycle already puts you several cell time constants past the step, where there is little left to cancel.

That is also where the speed limit lives. Push the frequency up and the half cycle shrinks toward $R_uC_d$ ; the sampling instant slides back up the exponential, and the charging current stops being negligible well before it stops being cancellable. A cell with a large $R_u$ or a large $C_d$ cannot be run fast, however clever the waveform. This is the constraint that set the shortest usable step time in [Lesson 7](L07-Resistance-DoubleLayer.ipynb) in new clothes, and it is why the figure above was recorded at an **ultramicroelectrode**: a small electrode has a small $C_d$ , and a small $C_d$ is what buys the frequency.

---
# Literature Examples

#### Dye sensitized solar cells

Dye sensitized solar cells rely upon the binding of molecular dyes to the surface of semiconductors. These dyes act as the photon absorbers that harvest the sun's energy. Photon absorption results in electronic excitation, and the excited electron is injected into the solid state semiconductor. Connection of this semiconductor with another electrode results in current flow. These devices offer all the benefits of molecular chemistry: the dyes can be tailored by exchange or installation of functional groups. 

The energy of the highest occupied molecular orbitals and the lowest unoccupied molecular orbitals relative to the band edges of the semiconductor are of critical importance. These energies are often analysed through voltammetry. For example, see [J. Mater. Chem. A 2014, 2, 16963](https://doi.org/10.1039/c4ta03226h).

<img src='../Resources/images/dssc-dye.png' width=30%>


<img src='../Resources/images/dssc-swv.png' width=40%>